In [ ]:
import os
import sys
import gc
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from tensorflow.keras import layers, models, Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

In [ ]:
%load_ext autoreload
%autoreload 2

# Pega o diretório atual e sobe um nível ('..')
project_root = os.path.abspath('..')

# Adiciona este diretório ao sys.path se ele ainda não estiver lá
if project_root not in sys.path:
    sys.path.append(project_root)

# --- Seus imports originais ---
import utils.processamento_dados as proc_dados
import utils.metricas_e_visualizacao as met_vil

In [ ]:
# --- 2. CONFIGURAÇÕES DE AMBIENTE E CAMINHOS ---
# Configuração básica de GPU (Memory Growth) para evitar alocação total imediata
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

In [ ]:
# DEFINA SEUS CAMINHOS AQUI:
BASE_DIR = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI"
data_3t_dir = os.path.join(BASE_DIR, "ADNI_3_4_NORMALIZED")
results_dir = os.path.join(BASE_DIR, "results", "fine_tuning_3t_k_fold")
os.makedirs(results_dir, exist_ok=True)

# Nome do modelo pré-treinado (Treinado no dataset 1.5T)
# pretrained_model_path = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/results_fit_15_predict_3/test_1/binary_classifier_noise_200_epochs_batch_15_2_classes.keras"

# Carregando último modelo treinado
latest_results = "/mnt/c/Users/Paulo Pires/Desktop/Alzheimer_cnn/ADNI/results_fit_15_predict_3"
n_results = len(os.listdir(latest_results))
latest_dir = os.path.join(latest_results, f"test_{n_results}")
pretrained_model_path = os.path.join(latest_dir, os.listdir(latest_dir)[0])

In [ ]:
# A pasta de resultados é organizada por pastas para cada teste
# Aqui, testamos quantas pastas já existem
n = len(os.listdir(results_dir))

# Caso já existam pastas, testamos se estas estão com conteúdo. Caso alguma não tenha conteúdo suficiente (treino foi interrompido
# e apenas parte dos resultados foi salva), nós apagamos a pasta "test_x" e recriamos.
if (n > 0):
    if (len(os.listdir(os.path.join(results_dir, f'test_{n}'))) < 3): 
        for item in os.listdir(os.path.join(results_dir, f"test_{n}")):
            os.remove(os.path.join(results_dir,  f"test_{n}", item))
        os.removedirs(os.path.join(results_dir, f'test_{n}'))
        n -= 1

# Criação da pasta
folder_name = f"test_{str(n+1)}"
results_dir = os.path.join(results_dir, folder_name)
os.makedirs(results_dir, exist_ok=True)
print(f"pasta {folder_name} criada")

In [ ]:
# === 4. PREPARAÇÃO DOS DADOS 3T (União de todas as pastas) ===
print("--- Carregando e Unificando Dados 3T ---")

dir_3t_train = os.path.join(data_3t_dir, "train")
dir_3t_val = os.path.join(data_3t_dir, "validation")
dir_3t_test = os.path.join(data_3t_dir, "test")

parts_X = []
parts_y = []

# Carrega tudo usando a função do seu módulo local
for d in [dir_3t_train, dir_3t_val, dir_3t_test]:
    print(f"Lendo pasta: {d}")
    try:
        x_part, y_part = proc_dados.load_nifti_data(d)
        if len(x_part) > 0:
            parts_X.append(x_part)
            parts_y.append(y_part)
    except Exception as e:
        print(f"Erro ao ler pasta {d}: {e}")

if len(parts_X) > 0:
    X_3t_all = np.concatenate(parts_X, axis=0)
    y_3t_all = np.concatenate(parts_y, axis=0)
    print(f"Total de dados 3T para K-Fold: {X_3t_all.shape}")
else:
    raise ValueError("Não foram encontrados dados nas pastas 3T.")

del parts_X, parts_y

# === 5. CONFIGURAÇÃO DO K-FOLD E MODELO ===
BATCH_SIZE = 8
EPOCHS = 50
K_FOLDS = 5
LEARNING_RATE = 0.001
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

# Listas para armazenar métricas de cada fold
fold_accuracies = []
fold_aucs = []
all_true_labels = []
all_pred_labels = []

print(f"\n>>> INICIANDO FINE-TUNING (K-FOLD = {K_FOLDS}) <<<")
print(f"Modelo Base para carregar pesos: {pretrained_model_path}")
print("Estratégia: Congelar 1º Bloco Convolucional\n")

# Verifica se o modelo base existe antes de começar o loop
if not os.path.exists(pretrained_model_path):
    raise FileNotFoundError(f"Modelo pré-treinado não encontrado em: {pretrained_model_path}")

In [ ]:
# FUNÇÕES AUXILIARES (VISUALIZAÇÃO)
def plot_training_history_binary(history, directory, title='training_history.png'):
    plt.figure(figsize=(12, 4))

    # Plot Loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Loss Graphic')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    # Plot Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy Graphic')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    save_path = os.path.join(directory, title)
    plt.savefig(save_path)
    plt.close() # Importante fechar para liberar memória da figura
    print(f"   Grafico de historico salvo em: {save_path}")

def save_confusion_matrix(y_true, y_pred, directory, fold_num):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['CN', 'AD'])
    
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(cmap=plt.cm.Blues, ax=ax, values_format='d')
    ax.set_title(f"Confusion Matrix - Fold {fold_num}")
    
    save_path = os.path.join(directory, f"confusion_matrix_fold_{fold_num}.png")
    plt.savefig(save_path)
    plt.close()
    print(f"   Matriz de confusão salva em: {save_path}")

# FUNÇÕES AUXILIARES (Augmentation & Generator)
def augment_and_balance_in_ram(X_full, y_full, indices):
    # 1. Pega labels deste fold para contagem
    y_subset = y_full[indices]
    
    # 2. Identifica classe minoritária
    counts = np.bincount(y_subset)
    minority_class = np.argmin(counts)
    majority_class = np.argmax(counts)
    n_minority = counts[minority_class]
    n_majority = counts[majority_class]
    
    print(f"   > Balanceamento: Majoritária ({majority_class}): {n_majority} | Minoritária ({minority_class}): {n_minority}")
    
    # 3. Calcula tamanho final:
    # Mantém todos da majoritária e multiplica a minoritária por 4 (Orig + 3 augs)
    # Se quiser balancear perfeitamente, ajuste a lógica aqui.
    total_final = n_majority + (n_minority * 4)
    
    input_shape = X_full[0].shape
    # Adiciona canal extra se não existir (D, H, W, 1)
    final_shape = (total_final, *input_shape, 1)
    
    print(f"   > Alocando memória para {total_final} amostras (float16)...")
    
    # 4. Alocação (Float16 é o segredo para não estourar RAM)
    try:
        X_aug = np.empty(final_shape, dtype=np.float16) 
        y_aug = np.empty((total_final,), dtype=np.int16)
    except MemoryError:
        print("ERRO FATAL: Memória RAM insuficiente para Augmentation.")
        raise
        
    # 5. Preenchimento e Augmentation
    cursor = 0
    total_indices = len(indices)
    
    # OBS: Certifique-se que proc_dados.augment_zoom/shift/rotation retornam array do mesmo tamanho
    for idx, i in enumerate(indices):
        orig = X_full[i] 
        lbl = y_full[i]
        
        # Garante dimensão do canal (H,W,D) -> (H,W,D,1)
        if len(orig.shape) == 3:
            orig_expanded = np.expand_dims(orig, axis=-1)
        else:
            orig_expanded = orig

        if lbl == majority_class:
            # Classe Majoritária: Copia apenas o original
            X_aug[cursor] = orig_expanded
            y_aug[cursor] = lbl
            cursor += 1
        else:
            # Classe Minoritária: Original + 3 Augmentations
            
            # 1. Original
            X_aug[cursor] = orig_expanded
            y_aug[cursor] = lbl
            cursor += 1
            
            # 2. Zoom
            zoom_img = proc_dados.augment_zoom(orig) # Sua função de zoom
            X_aug[cursor] = np.expand_dims(zoom_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
            # 3. Shift
            shift_img = proc_dados.augment_shift(orig) # Sua função de shift
            X_aug[cursor] = np.expand_dims(shift_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
            # 4. Rotation
            rot_img = proc_dados.augment_rotation(orig) # Sua função de rotação
            X_aug[cursor] = np.expand_dims(rot_img, axis=-1)
            y_aug[cursor] = lbl
            cursor += 1
            
        if idx % 100 == 0:
            print(f"    Processando: {idx}/{total_indices}...", end='\r')
            
    print(f"\nDados Augmentados na RAM! Shape: {X_aug.shape}")
    
    # 6. Embaralhar os dados gerados para o treino não ficar viciado na ordem
    shuf_idxs = np.arange(total_final)
    np.random.shuffle(shuf_idxs)
    
    return X_aug[shuf_idxs], y_aug[shuf_idxs]

def numpy_generator(x_data, y_data):
    for i in range(len(x_data)):
        yield x_data[i].astype(np.float16), y_data[i]

# Itera sobre os dados unificados (X_3t_all)
for fold_idx, (train_index, val_index) in enumerate(skf.split(X_3t_all, y_3t_all)):
    print(f"\n{'='*40}")
    print(f"INICIANDO FOLD {fold_idx+1}/{K_FOLDS} (FINE-TUNING COM AUGMENTATION)")
    print(f"{'='*40}")

    fold_dir = os.path.join(results_dir, f"fold_ft_{fold_idx+1}")
    os.makedirs(fold_dir, exist_ok=True)

    # --- A. PREPARAÇÃO DOS DADOS ---
    
    # VALIDAÇÃO: 
    X_val_fold = X_3t_all[val_index]
    if len(X_val_fold.shape) == 3:
        X_val_fold = np.expand_dims(X_val_fold, axis=-1)
    X_val_fold = X_val_fold.astype(np.float16)
    y_val_fold = y_3t_all[val_index]

    # TREINO: 
    # Aplicamos a função de Augmentation na RAM
    print("⏳ Gerando Augmentation e Balanceamento na RAM...")
    try:
        X_train_aug, y_train_aug = augment_and_balance_in_ram(X_3t_all, y_3t_all, train_index)
    except MemoryError:
        print("FATAL: Memória insuficiente no augmentation. Parando execução.")
        break
    except Exception as e:
        print(f"Erro no augmentation: {e}")
        break

    # --- B. CONFIGURAR GENERATORS (TF.DATA) ---
    print("Configurando tf.data Generators...")
    
    # Especificações dos tensores
    # Treino é float16 (conforme saído do augment_and_balance)
    train_spec_img = tf.TensorSpec(shape=X_train_aug.shape[1:], dtype=tf.float16) 
    train_spec_lbl = tf.TensorSpec(shape=(), dtype=tf.int16)
    
    # Validação é float16
    val_spec_img = tf.TensorSpec(shape=X_val_fold.shape[1:], dtype=tf.float16)
    val_spec_lbl = tf.TensorSpec(shape=(), dtype=tf.int16)

    # Dataset Treino (Generator converte f16 -> f32 on-the-fly)
    train_dataset = tf.data.Dataset.from_generator(
        lambda: numpy_generator(X_train_aug, y_train_aug),
        output_signature=(train_spec_img, train_spec_lbl)
    )
    # Shuffle buffer e Prefetch
    train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # Dataset Validação
    val_dataset = tf.data.Dataset.from_generator(
        lambda: numpy_generator(X_val_fold, y_val_fold),
        output_signature=(val_spec_img, val_spec_lbl)
    )
    val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    # --- C. MODELO E TRANSFER LEARNING ---
    print(f"Carregando modelo base de: {pretrained_model_path}")
    input_shape = X_train_aug.shape[1:] # ex: (156, 195, 160, 1)

    # 2. Carrega pesos
    try:
        model_ft = load_model(pretrained_model_path)
        print("Pesos carregados com sucesso.")
    except Exception as e:
        print(f"ERRO CRÍTICO ao carregar pesos: {e}")
        break

    # 3. Congela camadas iniciais (Bloco 1 e 2)
    # Ajuste o range se quiser congelar mais ou menos camadas
    print("   Congelando camadas iniciais (0 a 12)...")
    for i in range(13):
        model_ft.layers[i].trainable = False

    # 4. Compila
    model_ft.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    
    # Callbacks
    model_name = f"model_ft_fold_{fold_idx+1}.keras"
    model_save_path = os.path.join(fold_dir, model_name)
    
    callbacks_list = [
        ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=0),
        EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5)
    ]

    # --- D. TREINAMENTO ---
    print("🚀 Iniciando Fine-Tuning...")
    history = model_ft.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=val_dataset,
        callbacks=callbacks_list,
        verbose=1
    )
    
    # Salva gráfico histórico
    try: met_vil.plot_training_history_binary(history, fold_dir)
    except: pass

    # --- E. AVALIAÇÃO E PREDIÇÃO ---
    print("Carregando melhor modelo do fold para avaliação...")
    best_model = models.load_model(model_save_path)
    
    # Avaliação numérica
    loss, acc, auc = best_model.evaluate(val_dataset, verbose=0)
    fold_accuracies.append(acc)
    fold_aucs.append(auc)
    print(f"   Resultado Fold {fold_idx+1}: Acc={acc:.4f}, AUC={auc:.4f}")

    # Predição (Usando o gerador da validação para não estourar RAM)
    # model.predict(val_dataset) retorna na ordem correta do generator
    preds_prob = best_model.predict(val_dataset, verbose=0)
    
    # Binariza (Threshold 0.5)
    pred_labels = (preds_prob > 0.5).astype(int).flatten()
    true_labels = y_val_fold # Já temos na RAM
    
    # Garante shapes compatíveis (caso o batching do generator corte algo, embora raro no predict)
    limit = min(len(pred_labels), len(true_labels))
    pred_labels = pred_labels[:limit]
    true_labels = true_labels[:limit]

    # Relatórios
    met_vil.get_classification_report(true_labels, pred_labels, fold_dir, f'report_ft_fold_{fold_idx+1}')
    met_vil.plot_confusion_matrix(true_labels, pred_labels, fold_dir, f'conf_matrix_ft_fold_{fold_idx+1}', ['cn', 'ad'])

    # Acumula para relatório global
    all_true_labels.extend(true_labels)
    all_pred_labels.extend(pred_labels)

    # --- F. LIMPEZA DE MEMÓRIA ---
    print("🧹 Limpando memória...")
    del X_train_aug, y_train_aug, X_val_fold, y_val_fold
    del train_dataset, val_dataset, model_ft, best_model, history
    tf.keras.backend.clear_session()
    gc.collect()

In [ ]:
# ==========================================
# 4. RESULTADOS FINAIS AGREGADOS
# ==========================================
print("\n=== FINE-TUNING FINALIZADO ===")
print(f"Acurácia Média: {np.mean(fold_accuracies):.4f} (+/- {np.std(fold_accuracies):.4f})")
print(f"AUC Média: {np.mean(fold_aucs):.4f} (+/- {np.std(fold_aucs):.4f})")

# Gera Matriz Global
final_true = np.array(all_true_labels)
final_pred = np.array(all_pred_labels)

if len(final_true) > 0:
    met_vil.plot_confusion_matrix(
        final_true, 
        final_pred, 
        results_dir, 
        'FINAL_AGGREGATED_CONFUSION_MATRIX_FT', 
        ['cn', 'ad']
    )
    
    met_vil.get_classification_report(
        final_true, 
        final_pred, 
        results_dir, 
        'FINAL_AGGREGATED_REPORT_FT'
    )